<a href="https://colab.research.google.com/github/Trista1208/advanced_genAI/blob/main/Subsample_Multi_agent_system_design_Step_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **SECTION 1: IMPORTS**

In [1]:
# Installation
!pip install -q langdetect nltk
!pip install rank_bm25
!pip install -q langchain langchain-community langchain-core langchain-huggingface chromadb
!pip install pytrec_eval
!pip install -U transformers accelerate

# Standard library
import pickle
import os
import collections
import pathlib
import time
import json
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple
import functools
import numpy as np
import torch
import nltk
from langdetect import detect
from rank_bm25 import BM25Okapi
import re

# ML/Transformers
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    AutoModelForCausalLM
)

# LangChain
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Evaluation
import pytrec_eval
import pandas as pd
from tqdm import tqdm

# Google Colab
from google.colab import drive
import importlib.machinery
import importlib.util
import importlib
import sys
from importlib.machinery import SourceFileLoader

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you ha

## **SECTION 2: ENVIRONMENT SETUP**

In [2]:
# Mount Google Drive
drive.mount("/content/drive")

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Download NLTK data
nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

# Stopwords
STOP_EN = set(nltk.corpus.stopwords.words("english"))
STOP_DE = set(nltk.corpus.stopwords.words("german"))

Mounted at /content/drive
Using device: cuda


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## **SECTION 3: PATHS**


Defines file paths for all data sources (pickled retrievers, QA benchmarks, relevance judgments) and sets evaluation metrics (P@5, P@10, recall@100, reciprocal rank, NDCG@10).

In [3]:
ROOT = pathlib.Path("/content/drive/MyDrive/Adv_GenAI").resolve()

PATH_BM25_PICKLE = "/content/drive/MyDrive/Adv_GenAI/storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"
PATH_QA = ROOT / "benchmark/benchmark_qa_bilingual.json"
PATH_QRELS_FIXED = pathlib.Path("/content/drive/MyDrive/Adv_GenAI/benchmark/score/fixed_size")

# Evaluation metrics
METRICS = {"P_5", "P_10", "recall_100", "recip_rank", "ndcg_cut_10"}

## **SECTION 4: TRANSLATION COMPONENT**

Implements bilingual EN↔DE translator using Facebook's M2M100 model with caching for efficient cross-language query translation in the retrieval pipeline.

In [4]:
class EnDeTranslator:
    """Greedy EN⇄DE translation (cached – 1st call downloads weights)."""

    def __init__(self, model="facebook/m2m100_418M", device: str | None = None) -> None:
        self.device = device or DEVICE
        self.tok = M2M100Tokenizer.from_pretrained(model)
        self.model = M2M100ForConditionalGeneration.from_pretrained(model).to(self.device)

    @functools.lru_cache(maxsize=512)
    def translate(self, text: str, tgt: str) -> str:
        src = detect(text) if text.strip() else "en"
        src = src if src in ("en", "de") else "en"
        if src == tgt:
            return text
        self.tok.src_lang = src
        ids = self.tok(text, return_tensors="pt").to(self.device)
        out = self.model.generate(**ids,
                                   forced_bos_token_id=self.tok.get_lang_id(tgt),
                                   num_beams=1, max_new_tokens=128)
        return self.tok.decode(out[0], skip_special_tokens=True)


# Initialize translator
translator = EnDeTranslator()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

## **SECTION 5: QUERY EXPANSION**

Implements pseudo-relevance feedback (PRF) by extracting frequent terms from top-k initial results to expand and improve query representation.

In [5]:
def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
    """Simple pseudo-relevance feedback (token-frequency expansion)."""
    hits = base_retriever.search(query, top_k=fb_docs)
    tokens = [
        t.lower() for h in hits
        for t in nltk.word_tokenize(h.page_content.lower())
        if t.isalpha() and t not in STOP_EN and t not in STOP_DE
    ]
    extra = " ".join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
    return f"{query} {extra}" if extra else query

## **SECTION 6: BILINGUAL BM25 RETRIEVAL**

Creates dual BM25 indices for English and German documents with automatic query translation, returning deduplicated results ranked by BM25 scores across both languages.

In [6]:
class BilingualBM25:
    """Two BM25Okapi indices (EN / DE) with optional query translation."""

    def __init__(self, docs: List[Document]) -> None:
        self.docs_by_lang = {"en": [], "de": []}
        self.toks_by_lang = {"en": [], "de": []}
        for d in docs:
            lang = d.metadata.get("language", "en")
            lang = lang if lang in ("en", "de") else "en"
            self.docs_by_lang[lang].append(d)
            self.toks_by_lang[lang].append(nltk.word_tokenize(d.page_content))
        self.bm25 = {l: BM25Okapi(tok) for l, tok in self.toks_by_lang.items() if tok}

    def _rank_lang(self, q: str, lang: str, k: int) -> List[Document]:
        scores = self.bm25[lang].get_scores(nltk.word_tokenize(q))
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata["bm25_score"] = float(scores[i])
            hits.append(d)
        return hits

    def search(self, query: str, top_k: int = 100) -> List[Document]:
        src = detect(query) if query.strip() else "en"
        src = src if src in ("en", "de") else "en"
        bag = []
        for lang in ("en", "de"):
            q_lang = translator.translate(query, lang) if lang != src else query
            bag.extend(self._rank_lang(q_lang, lang, top_k))
        # deduplicate by record/chunk id (keep highest score)
        best: Dict[str, Document] = {}
        for d in bag:
            uid = d.metadata.get("chunk_id") or d.metadata.get("record_id")
            if uid not in best or d.metadata["bm25_score"] > best[uid].metadata["bm25_score"]:
                best[uid] = d
        return sorted(best.values(), key=lambda d: d.metadata["bm25_score"], reverse=True)[:top_k]


class QEBM25:
    """BM25 + PRF wrapper"""

    def __init__(self, base: BilingualBM25) -> None:
        self.base = base

    def search(self, query: str, top_k: int = 100) -> List[Document]:
        return self.base.search(_expand_query(query, self.base), top_k)

## **SECTION 7: LOAD RETRIEVERS**

Loads three pre-built retrieval systems: BM25 with query expansion, Dense retriever (Multilingual E5 embeddings), and GraphRAG retriever from disk/modules.

In [7]:
# Load BM25 retriever & attach translator
with open(PATH_BM25_PICKLE, "rb") as fh:
    bm25_fixed_qe = pickle.load(fh)
bm25_fixed_qe.base.translator = translator  # restore translation ability

# Load Dense retriever
dense_loader = importlib.machinery.SourceFileLoader(
    "dense_mod", str(PATH_DENSE_LOADER)).load_module()
dense_fixed = dense_loader.load_dense_fixed(device=DEVICE, k=100)

# Load GraphRAG retriever
grag_loader = importlib.machinery.SourceFileLoader(
    "grag_mod", str(PATH_GRAG_LOADER)).load_module()
graph_rag = grag_loader  # public API: graph_rag.retrieve()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## **SECTION 8: Query Classifier Agent**

The Query Classifier Agent supports the confidence orchestration by classifying the query into factoid, semantic and hybrid queries. We refrain from a more sophisticated prompt as the `google/flan-t5-base` doesn't allow for a longer prompt. But allowing a fast classifying of the queries.

In [8]:
from pydantic import BaseModel, Field
from typing import Literal
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class QueryClassifierAgent:
    """Lightweight query classifier using flan-t5."""

    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        max_new_tokens: int = 32,
        device: str = DEVICE,
    ):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
        self.max_new_tokens = max_new_tokens

    def _build_prompt(self, query: str) -> str:
        """Short prompt that flan-t5 can handle."""
        return f"""Classify as FACTOID, SEMANTIC, or HYBRID.

FACTOID = names, numbers, dates, who/when/where
SEMANTIC = why, how, explain, conceptual
HYBRID = mix of both

Query: {query}

Answer:"""

    def classify(self, query: str):
        """Main entry point: classify query and return analysis."""
        prompt = self._build_prompt(query)

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)



        return response

## **SECTION 9: ORCHESTRATION STRATEGIES**

Implements three multi-agent orchestration strategies:

*   Waterfall strategy combining parallel retrieval with conditional fallback
*   Voting strategy based on weighted rank fusion across retrievers

*   Confidence-based strategy that dynamically adjusts retriever weights according to query features.




In [20]:
import math
from collections import defaultdict

def _uid(doc):
    return doc.metadata.get("chunk_id") or doc.metadata.get("record_id")

def _safe_unique(docs):
    """Dedup by uid, keep first occurrence."""
    out = []
    seen = set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out

def _rrf_fuse(runs: dict, k_rrf: int = 60, weights: dict | None = None):
    """
    runs: {"bm25": [docs...], "dense": [docs...], "graph": [docs...]}
    Returns: docs sorted by weighted RRF score.
    """
    weights = weights or {"bm25": 1.0, "dense": 1.0, "graph": 0.7}
    scores = defaultdict(float)
    store = {}

    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w * (1.0 / (k_rrf + rank))

    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata["fused_score"] = float(scores[_uid(d)])
    return fused

class SimpleOverlapReranker:
    def rerank(self, docs, query: str, top_k: int = 10):
        q_terms = set(t.lower() for t in query.split() if t.strip())
        scored = []
        for d in docs:
            text = (d.metadata.get("original_text") or d.page_content or "").lower()
            d_terms = set(text.split())
            overlap = len(q_terms & d_terms) / max(len(q_terms), 1)
            scored.append((overlap, d))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [d for _, d in scored[:top_k]]

def orchestrate_parallel_fusion(
    query: str,
    top_k: int = 5,
    pre_k: int | None = None,
    use_graph: bool = True,
    weights: dict | None = None,
    apply_overlap_rerank: bool = False,
):

    trace = []
    pre_k = pre_k or max(30, top_k * 10)

    bm25_docs = bm25_fixed_qe.search(query, top_k=pre_k)
    trace.append(f"BM25 retrieved {len(bm25_docs)}")

    dense_docs = dense_fixed.search(query, top_k=pre_k)
    trace.append(f"Dense retrieved {len(dense_docs)}")

    graph_docs = []
    if use_graph:
        graph_docs = graph_rag.retrieve(query, top_k=pre_k)
        trace.append(f"GraphRAG retrieved {len(graph_docs)}")

    bm25_docs = _safe_unique(bm25_docs)
    dense_docs = _safe_unique(dense_docs)
    graph_docs = _safe_unique(graph_docs)

    # Fuse
    fused = _rrf_fuse(
        runs={"bm25": bm25_docs, "dense": dense_docs, "graph": graph_docs},
        k_rrf=60,
        weights=weights or {"bm25": 1.2, "dense": 1.0, "graph": 0.6},
    )
    trace.append("Fusion: weighted RRF")

    # Optional rerank for precision
    if apply_overlap_rerank:
        rer = SimpleOverlapReranker()
        fused = rer.rerank(fused[:max(50, top_k*10)], query, top_k=max(50, top_k*10))
        trace.append("Post-rerank: overlap")

    return fused[:top_k], trace



def waterfall_orchestrate(query: str, top_k: int = 5):
    """
    GraphRAG is only enabled if fused is too weak in terms of overlap
    """
    docs, trace = orchestrate_parallel_fusion(
        query=query,
        top_k=top_k,
        pre_k=max(30, top_k * 10),
        use_graph=False,
        weights={"bm25": 1.2, "dense": 1.0, "graph": 0.0},
        apply_overlap_rerank=False,
    )

    q_terms = set(t.lower() for t in query.split() if t.strip())
    top_text = (docs[0].metadata.get("original_text") or docs[0].page_content or "").lower() if docs else ""
    overlap = len(q_terms & set(top_text.split())) / max(len(q_terms), 1)

    if overlap < 0.05:
        docs2, trace2 = orchestrate_parallel_fusion(
            query=query,
            top_k=top_k,
            pre_k=max(30, top_k * 10),
            use_graph=True,
            weights={"bm25": 1.1, "dense": 1.0, "graph": 0.7},
            apply_overlap_rerank=False,
        )
        trace += ["Fallback triggered: low overlap -> add GraphRAG"] + trace2
        return docs2, trace

    trace.append(f"Critic: overlap={overlap:.3f} (ok)")
    return docs, trace


def voting_orchestrate(query: str, top_k: int = 5):
    return orchestrate_parallel_fusion(
        query=query,
        top_k=top_k,
        pre_k=max(30, top_k * 10),
        use_graph=True,
        weights={"bm25": 1.2, "dense": 1.0, "graph": 0.6},
        apply_overlap_rerank=False,
    )


def confidence_orchestrate(query: str, top_k: int = 5):
    """
    We choose weights dynamically based on the type of query, classified by the QueryClassifierAgent.
    """

    classifier = QueryClassifierAgent()

    # classify query
    query_class = classifier.classify(query)


    if query_class == "FACTOID":
        w = {"bm25": 1.4, "dense": 0.9, "graph": 0.5}
        mode = "factoid -> BM25 boosted"
    elif query_class == "SEMANTIC":
        w = {"bm25": 0.9, "dense": 1.3, "graph": 0.6}
        mode = "semantic -> Dense boosted"
    # HYBRID and fallback if classifier fails
    else:
        w = {"bm25": 1.0, "dense": 1.1, "graph": 0.5}
        mode = "hybrid -> balanced"

    docs, trace = orchestrate_parallel_fusion(
        query=query,
        top_k=top_k,
        pre_k=max(30, top_k * 10),
        use_graph=True,
        weights=w,
        apply_overlap_rerank=False,
    )
    trace.insert(0, f"Confidence router: {mode}")
    return docs, trace


## **SECTION 10: RETRIEVER WRAPPERS FOR EVALUATION**

Wraps orchestration functions into retriever classes with standardized .search() interface for consistent evaluation across all strategies.

In [11]:
class WaterfallRetriever:
    name = "Waterfall"

    def search(self, query, top_k=100):
        docs, _ = waterfall_orchestrate(query, top_k=top_k)
        return docs


class VotingRetriever:
    name = "Voting"

    def search(self, query, top_k=100):
        docs, _ = voting_orchestrate(query, top_k=top_k)
        return docs


class ConfidenceRetriever:
    name = "Confidence"

    def search(self, query, top_k=100):
        docs, _ = confidence_orchestrate(query, top_k=top_k)
        return docs

## **SECTION 11: ANSWER SYNTHESIS AND ORCHESTRATION**

Implements the main orchestrator that integrates retrieval strategies with answer synthesis. Based on the selected orchestration strategy, the orchestrator retrieves relevant documents and invokes the Answer Synthesizer Agent to generate the final answer.

In [12]:
class Orchestrator:
    """Main orchestrator combining retrieval strategies and answer synthesis."""

    def __init__(self, bm25, dense, graph, synthesizer):
        self.bm25 = bm25
        self.dense = dense
        self.graph = graph
        self.synthesizer = synthesizer

    def run(self, strategy: str, query: str, top_k=5):
        if strategy == "waterfall":
            docs, trace = waterfall_orchestrate(query, top_k)
        elif strategy == "voting":
            docs, trace = voting_orchestrate(query, top_k)
        elif strategy == "confidence":
            docs, trace = confidence_orchestrate(query, top_k)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        answer = self.synthesizer.generate(query, docs)

        return {
            "query": query,
            "strategy": strategy,
            "trace": trace,
            "documents": docs,
            "answer": answer,
        }


## **SECTION 12: EVALUATION UTILITIES**

Provides functions to load relevance judgments (qrels), build TREC-format runs, compute macro-averaged metrics, measure efficiency (time/calls), and check answer presence in context.

In [21]:
def load_qrels(folder: pathlib.Path) -> dict:
    qrels = defaultdict(dict)
    for fp in folder.glob("*.json"):
        did = fp.stem
        for qid, pay in json.loads(fp.read_text()).items():
            if pay["relevance_score"] >= 0.5:
                qrels[qid][did] = 1
    return qrels


VARIANTS = {
    "BM25": bm25_fixed_qe,
    "Dense": dense_fixed,
    "Waterfall": WaterfallRetriever(),
    "Voting": VotingRetriever(),
    "Confidence": ConfidenceRetriever(),
}

def build_runs_orchestration():
    runs = {}

    for name, retr in VARIANTS.items():
        run = defaultdict(dict)

        for q in tqdm(qa_data, desc=name):
            qid = str(q["id"])
            docs = retr.search(q["question"], top_k=100)

            for rank, d in enumerate(docs, start=1):
                did = d.metadata.get("chunk_id") or d.metadata.get("record_id")
                if did is None:
                    continue
                run[qid][did] = 100 - rank

        runs[name] = run

    return runs



def macro(run, qrels):
    df = pd.DataFrame(
        pytrec_eval.RelevanceEvaluator(qrels, METRICS).evaluate(run)
    ).T
    return df.mean()


def measure_efficiency(orchestrator_fn, questions, top_k=100):
    """Measure time and retriever calls for an orchestration strategy."""
    times = []
    calls = []

    for q in questions:
        start = time.time()
        _, trace = orchestrator_fn(q, top_k=top_k)
        times.append(time.time() - start)
        calls.append(len(trace))

    return {
        "avg_time": sum(times) / len(times),
        "avg_calls": sum(calls) / len(calls),
    }


def answer_in_context(docs, answer):
    ctx = " ".join(d.page_content.lower() for d in docs)
    return answer.lower() in ctx

## **SECTION 13: TEST SINGLE QUERY**

Quick sanity check: runs one example query through all three retrievers (BM25, Dense, GraphRAG) and displays result types, metadata, and preview snippets.

In [15]:
QUERY = "Who at ETH received ERC grants?"
TOP_K = 5

bm25_res = bm25_fixed_qe.search(QUERY, top_k=TOP_K)
print(type(bm25_res))
print(len(bm25_res))
print(type(bm25_res[0]))
print(bm25_res[0].metadata.keys())

dense_res = dense_fixed.search(QUERY, top_k=TOP_K)
print(type(dense_res))
print(len(dense_res))
print(type(dense_res[0]))
print(dense_res[0].metadata.keys())

graph_res = graph_rag.retrieve(QUERY, top_k=TOP_K)
print(type(graph_res))
print(len(graph_res))
print(type(graph_res[0]))
print(graph_res[0].metadata.keys())

print("BM25 top text:\n", bm25_res[0].page_content[:300])
print("\nDense top text:\n", dense_res[0].page_content[:300])
print("\nGraphRAG top text:\n", graph_res[0].page_content[:300])

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<class 'list'>
5
<class 'langchain_core.documents.base.Document'>
dict_keys(['year', 'text_stats', 'source', 'title', 'chunk_id', 'month', 'doc_id', 'chunk_token_count', 'domain', 'filename', 'language', 'grant_type', 'chunk_summary', 'document_type', 'role_annotations', 'content_year', 'entities', 'content_month', 'numeric_facts', 'department', 'initiative', 'event_dates', 'topic_tags', 'record_id', 'original_text', 'bm25_score'])
<class 'list'>
5
<class 'langchain_core.documents.base.Document'>
dict_keys(['language', 'entities', 'document_type', 'content_year', 'chunk_summary', 'numeric_facts', 'filename', 'source', 'record_id', 'role_annotations', 'year', 'department', 'content_month', 'topic_tags', 'month', 'grant_type', 'initiative', 'title', 'text_stats', 'event_dates', 'chunk_token_count', 'domain', 'chunk_id', 'doc_id', 'original_text', 'dense_score'])
<class 'list'>
5
<class 'langchain_core.documents.base.Document'>
dict_keys(['year', 'text_stats', 'source', 'title', 'chunk_id

## **SECTION 14: LOAD QA BENCHMARK DATA**

Loads bilingual question-answer pairs from JSON, extracts both English and German versions, creating parallel test sets for evaluation.

In [16]:
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print("Loaded QA pairs:", len(qa_data))
print(qa_data[0])
print(qa_data[0].keys())

questions = []
answers = []

for x in qa_data:
    questions.append(x["question"])
    answers.append(x["answer"])
    questions.append(x["question_de"])
    answers.append(x["answer_de"])

for i in range(3):
    print("Q:", questions[i])
    print("A:", answers[i])
    print()

Loaded QA pairs: 25
{'id': 1, 'question': 'who was president of eth in 2003?', 'answer': 'olaf kübler', 'answer_de': 'olaf kuebler', 'question_de': 'wer war 2003 praesident von eth?'}
dict_keys(['id', 'question', 'answer', 'answer_de', 'question_de'])
Q: who was president of eth in 2003?
A: olaf kübler

Q: wer war 2003 praesident von eth?
A: olaf kuebler

Q: who were the rectors of eth between 2017 and 2022?
A: sarah springman, günther dissertori.



# **SECTION 15: SIMPLE HIT RATE TEST**

Evaluates semantic answer coverage by counting how often the retrieved context contains tokens from the gold answer, using a soft overlap criterion across all queries and orchestration strategies.

In [17]:
# Normalization

def normalize(text: str):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return set(text.split())

def answer_in_context_soft(docs, answer, min_overlap=2):
    ans_tokens = normalize(answer)

    for d in docs:
        doc_text = d.metadata.get("original_text") or d.page_content
        doc_tokens = normalize(doc_text)

        if len(ans_tokens & doc_tokens) >= min_overlap:
            return 1
    return 0

# Evaluation on ALL questions

wf_hits = 0
vote_hits = 0
conf_hits = 0

total = len(questions)

for q, a in tqdm(
    zip(questions, answers),
    total=total,
    desc="Answer-in-context (ALL questions)",
):
    wf_docs, _ = waterfall_orchestrate(q)
    vote_docs, _ = voting_orchestrate(q)
    conf_docs, _ = confidence_orchestrate(q)

    wf_hits += answer_in_context_soft(wf_docs, a)
    vote_hits += answer_in_context_soft(vote_docs, a)
    conf_hits += answer_in_context_soft(conf_docs, a)

# Results

print("RESULTS (ALL questions):")
print(f"Waterfall hit rate:  {wf_hits / total:.3f}")
print(f"Voting hit rate:     {vote_hits / total:.3f}")
print(f"Confidence hit rate: {conf_hits / total:.3f}")


Answer-in-context (ALL questions):   0%|          | 0/50 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Answer-in-context (ALL questions): 100%|██████████| 50/50 [02:33<00:00,  3.07s/it]

RESULTS (ALL questions):
Waterfall hit rate:  0.880
Voting hit rate:     0.900
Confidence hit rate: 0.920


## **SECTION 16: LOAD QRELS FOR FORMAL EVALUATION**

Loads official relevance judgments for fixed-size and semantic chunking approaches to enable formal IR metrics evaluation.

In [18]:
QRELS = {
    "fixed": load_qrels(PATH_QRELS_FIXED)
}

print(f"✓ Qrels loaded fixed_size – {len(QRELS['fixed'])} queries with judgements")

✓ Qrels loaded fixed_size – 24 queries with judgements



## **SECTION 17: BUILD RUNS & EVALUATE**

Generates TREC-format retrieval runs for all orchestration strategies and computes standard IR metrics (P@5, P@10, recall, NDCG, MRR) using pytrec_eval.

In [22]:
VARIANTS = {
    "Waterfall": WaterfallRetriever(),
    "Voting": VotingRetriever(),
    "Confidence": ConfidenceRetriever(),
}

runs = build_runs_orchestration()

tbl = (
    pd.concat(
        {k: macro(r, QRELS["fixed"]) for k, r in runs.items()},
        axis=1
    )
    .T.round(3)
)

display(tbl)

Confidence: 100%|██████████| 25/25 [01:00<00:00,  2.42s/it]


,recip_rank,P_5,P_10,recall_100,ndcg_cut_10
Waterfall,0.447,0.308,0.296,0.283,0.311
Voting,0.421,0.325,0.342,0.298,0.350
Confidence,0.403,0.325,0.321,0.298,0.329


## **SECTION 18: EFFICIENCY ANALYSIS**

Measures computational efficiency of each orchestration strategy by tracking average query time and number of retriever calls, with example execution traces.

In [23]:
eff_waterfall = measure_efficiency(waterfall_orchestrate, questions)
eff_voting = measure_efficiency(voting_orchestrate, questions)
eff_confidence = measure_efficiency(confidence_orchestrate, questions)

print("Waterfall:", eff_waterfall)
print("Voting:", eff_voting)
print("Confidence:", eff_confidence)

# Show example traces
_, trace = waterfall_orchestrate(questions[0])
print(trace)

_, trace = voting_orchestrate(questions[0])
print(trace)

Waterfall: {'avg_time': 0.30108607292175293, 'avg_calls': 4.4}
Voting: {'avg_time': 0.2996247482299805, 'avg_calls': 4.0}
Confidence: {'avg_time': 2.420973234176636, 'avg_calls': 5.0}
['BM25 retrieved 50', 'Dense retrieved 50', 'Fusion: weighted RRF', 'Critic: overlap=0.714 (ok)']
['BM25 retrieved 50', 'Dense retrieved 50', 'GraphRAG retrieved 50', 'Fusion: weighted RRF']


## **SECTION 19: QA EVALUATION (ANSWER SYNTHESIS)**

Assesses answer synthesis quality by measuring how often the generated answer overlaps with the gold answer across different orchestration strategies, providing an answer-level complement to retrieval evaluation. The answers are generated using an instruction-tuned LLM (Mistral-7B-Instruct).

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Extracts a small set of query-relevant sentences using token overlap to reduce context length before answer synthesis.

In [25]:
def split_sentences(text: str):
    return re.split(r'(?<=[.!?])\s+', text)

def sentence_overlap(sent: str, query: str):
    q = set(query.lower().split())
    s = set(sent.lower().split())
    return len(q & s)

def select_best_sentences(text: str, query: str, max_sentences=5):
    sents = split_sentences(text)
    scored = [
        (sentence_overlap(s, query), s)
        for s in sents
        if len(s) > 20
    ]
    scored.sort(reverse=True)
    return [s for _, s in scored[:max_sentences]]

This agent synthesizes a final answer by selecting a small set of query-relevant sentences from the top retrieved documents and generating a short, factual response using an instruction-tuned language model. If the answer is not supported by the retrieved context, it returns "NOT FOUND IN CONTEXT".

In [26]:
class AnswerSynthesizerAgent:
    def __init__(self, max_new_tokens=128):
        self.tokenizer = tokenizer
        self.model = model
        self.max_new_tokens = max_new_tokens

    def generate(self, query: str, docs):
        context_blocks = []

        for d in docs[:7]:
            raw = d.metadata.get("original_text") or d.page_content
            best_sents = select_best_sentences(raw, query, max_sentences=2)
            if best_sents:
                context_blocks.append(" ".join(best_sents))

        context = "\n".join(context_blocks)


        prompt = f"""[INST]
Answer the question using the context.
Give a short, factual answer.
If the answer is not clearly supported by the context, say "NOT FOUND IN CONTEXT".

Context:
{context}

Question: {query}
[/INST]
"""

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
            )

        decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # remove prompt prefix
        answer = decoded.split("[/INST]")[-1].strip()
        return answer

Instantiates the answer synthesizer and connects it to the orchestration pipeline.

In [28]:
synthesizer = AnswerSynthesizerAgent()

orch = Orchestrator(
    bm25=bm25_fixed_qe,
    dense=dense_fixed,
    graph=graph_rag,
    synthesizer=synthesizer,
)

Runs a qualitative sanity check by generating answers for a set of questions using different orchestration strategies. For each query, the generated answers are compared against the gold answers.

In [29]:
N = 50

for idx in range(N):
    query = questions[idx]
    gold = answers[idx]

    print("\n" + "#" * 80)
    print(f"QUESTION {idx}: {query}")
    print(f"GOLD ANSWER: {gold}")

    for strategy in ["waterfall", "voting", "confidence"]:
        res = orch.run(strategy, query, top_k=7)

        print("-" * 60)
        print("STRATEGY:", strategy.upper())
        print("GENERATED ANSWER:")
        print(res["answer"])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



################################################################################
QUESTION 0: who was president of eth in 2003?
GOLD ANSWER: olaf kübler


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


------------------------------------------------------------
STRATEGY: WATERFALL
GENERATED ANSWER:
NOT FOUND IN CONTEXT. The context does not provide information about who was the president of ETH Zurich in 2003.
------------------------------------------------------------
STRATEGY: VOTING
GENERATED ANSWER:
NOT FOUND IN CONTEXT. The context does not provide information about who was the president of ETH Zurich in 2003.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

------------------------------------------------------------
STRATEGY: CONFIDENCE
GENERATED ANSWER:
Yes, according to the context, Eth Zurich students are competing in the Olympics.

################################################################################
QUESTION 49: organisiert eth irgendwelche wettbewerbe?
GOLD ANSWER: eth zurich organisiert eine reihe von wettbewerben fuer das oeffentliche gut, adressieren junge, aufstrebende forscher, sport, innovatoren, und entrepreneurs. in welchem bereich moechten sie erkunden? nicht foerdern wettbewerbe. wohl, ein sehr beruehmter ist der cybathlon, der menzoned sein koennte, wenn der benutzer speziell fragt, zum beispiel im kontext der frage 24.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


------------------------------------------------------------
STRATEGY: WATERFALL
GENERATED ANSWER:
Yes, Eth Zurich organized a photo contest in 2023 as mentioned in the context.
------------------------------------------------------------
STRATEGY: VOTING
GENERATED ANSWER:
Yes, Eth Zurich organized a photo contest in 2023 as mentioned in the context.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


------------------------------------------------------------
STRATEGY: CONFIDENCE
GENERATED ANSWER:
Yes, Eth Zurich organizes competitions, specifically a photo contest for depicting the unique atmosphere and features of the Hönggerberg campus. (Context)
